# Deep Research Platform

## Components, Tools and Agents used
1. Clarifying agent: ask the provide clarity on the questions by providing some questions for the user to respond to
2. openai agent guardrail tools and agent to guide and prevent the planner agent from planning when the user provide out of scope area.
3. Planner ggent: provides the set of web searches to be done
4. Writer agent: Write the report of the search outcome
5. Notificatin agent: Push notification to the user when the research is done
6. Email agent: Send the full report of the search results to the user.
7. gradio: UI: compoent/tools which provides the interface for user interaction


In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool, input_guardrail, GuardrailFunctionOutput
from agents.model_settings import ModelSettings
from agents.exceptions import InputGuardrailTripwireTriggered
from pydantic import BaseModel, Field
from typing import List, Dict, Tuple
from dotenv import load_dotenv
import os
import requests
import asyncio
import gradio as gr


In [2]:
# Load environment
load_dotenv(override=True)


True

# Create an Agent to ask clarifying questions


In [3]:
INSTRUCTIONS_FOR_CLARIFYING_QUESTIONS = """
YOu are a research analyst. Given a research query, generate 3-5 clarifying questionsm that would help narrow down than improve the research.
Focus on:
- Scope and timeframe of the research
- specific aspects of the interested topic
- Intended use case or audience for the research
- Level of details needed

Make questions concise and easy to answer.
"""

class ClarifyingQuestion(BaseModel):
    question: str = Field(description="The clarifying question to ask the user")

    why_important: str = Field(description="A brief explanation of why this question is important")


class ClarifyingQuestions(BaseModel):
    questions: list[ClarifyingQuestion] = Field(description="A list of clarifying questions")


clarifying_agent = Agent(
    name="ClarifyingAgent",
    instructions=INSTRUCTIONS_FOR_CLARIFYING_QUESTIONS,
    model="gpt-4o-mini",
    output_type=ClarifyingQuestions,
)


# Create guardrail agent to limit the scope of user question

In [4]:
SCOPE_LIMIT_INSTRUCTIONS = """
You are a research scope guardrail. 
Given a research query, determine if it is within the allowed research scope.

OUT OF SCOPE topics include:
- Harmful, unethical, or illegal research
- Privacy invasion or misinformation promotion
- Weapons, drugs, or dangerous substances
- Content that is outside the intended use case or audience

Respond with:
- is_out_of_scope: True if the query is out of scope, False if it is allowed
- reason: A brief explanation of why the query is or isn't out of scope
"""

class ScopeLimit(BaseModel):
    is_out_of_scope: bool = Field(description="Whether the query is out of scope")
    reason: str = Field(description="A brief explanation of the decision")

scope_limiter_agent = Agent(
    name="ScopeLimiterAgent",
    instructions=SCOPE_LIMIT_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ScopeLimit,
)

@input_guardrail
async def guardrail_scope_limiter(ctx, agent, message):
    result = await Runner.run(scope_limiter_agent, message, context=ctx.context)
    output = result.final_output

    return GuardrailFunctionOutput(
        output_info={"Scope limit reason": output.reason},
        tripwire_triggered=output.is_out_of_scope
    )

# Search Agent


In [5]:
INSTRUCTIONS = """
    You are a research assistant. Given a search term, you search the web for that term and 
    produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300
    words. Capture the main points.
    Write succintly, no need to have complete sentences or good grammar.
    This will be consumed by someone synthesizing a report, so it's vital you capture the essence and ignore any fluff.
    Do not include any additional commentary other than the summary itself.
    """

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
    )

# Planner Agent

In [6]:
How_MANY_SEARCHES = 3

PLANNER_INSTRUCTIONS = """
You are a helpful research assistant. Given a query and optional clarifications,
come up with a set of web searches to perform to best answer the query, Output {HOW_MANY_SEARCHES} terms to query for
If clarification are provided, use them to make your searches more targeted and relevant.
"""

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searched to perform to best answer the query.")


# Create the planner agent
planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_INSTRUCTIONS,
    model="gpt-4o-mini",
    input_guardrails=[guardrail_scope_limiter],
    output_type=WebSearchPlan,
)



Create Notification tool and  Agent


In [7]:
import re

NOTIFICATION_INSTRUCTIONS = """
You are able to send a research report notification via Pushover.
You will be provided with a detailed markdown report. You should use your tool to send one notification 
with the report content. The notification should be clean and readable.
"""


@function_tool
def send_result_notification(result_report: str) -> Dict[str, str]:
    """ Send a resarch report as pushover notifcation """
    url = "https://api.pushover.net/1/messages.json"

    clean_content = re.sub(r'#{1,6}\s*', '', result_report)  
    clearn_content = re.sub(r'\*\*(.*?)\*\*', r'\1', result_report)
    clean_content = re.sub(r'\*(.*?)\*', r'\1', result_report) 
    clean_content = re.sub(r'\n\n+', '\n\n', clean_content)

    if len(clean_content) > 330:
        clean_content = clean_content[:350] + "...\n\n[Report truncated - see full report in email]"

        data = {
            "token": os.getenv("PUSHOVER_APP_TOKEN"),
            "user": os.getenv("PUSHOVER_USER_KEY"),
            "title": "Research Report",
            "message": clean_content,
        }

        response = requests.post(url, data=data)
        if response.status_code == 200:
            return {"sstatus": "success", "message": "Notification sent successfully"}
        else:
            return {"status": "error", "message": f"Error {response.status_code}: {response.text}"}

notification_agent = Agent(
    name="NotificationAgent",
    instructions=NOTIFICATION_INSTRUCTIONS,
    model="gpt-4o-mini",
    tools=[send_result_notification],
)



Create Email Agent

In [8]:
import resend


INSTRUNCTIONS = """
You are a helpful assistant that can send out a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line.
"""

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:

    """ Send a research report as an HTML email using Resend """
    resend.api_key = os.getenv("RESEND_API_KEY")

    try:
        resend.Emails.send({
            "from": os.getenv("EMAIL_FROM"),
            "to": [os.getenv("EMAIL_TO")],
            "subject": subject,
            "html": html_body,
        })
        return {"status": "success", "message": f"Email sent successfully"}
    except Exception as e:
        return {"status": "error", "message": f"Error sending email: {str(e)}"}


email_agent = Agent(
    name="EmailAgent",
    instructions=INSTRUNCTIONS,
    model="gpt-4o-mini",
    tools=[send_html_email],
)



Writer Agent

In [9]:
# Writer Agent
WRITER_INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, any clarifications provided by the user, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="A list of 3-5 suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=WRITER_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)


# Research Manager

In [10]:
async def get_clarifying_questions(query: str):
    """ Use the clarifying agent to generate questions about the query"""
    print("Generating clarifying questions...")
    result = await Runner.run(clarifying_agent, f"Research query: {query}")
    print(f"Generated {len(result.final_output.questions)} questions")
    return result.final_output

async def plan_searches(query: str, clarifications: str = ""):
    """Use the planner agent to plan which searches to rin for the query"""
    print("Planning searches...")
    input_text = f"Query: {query}"
    if clarifications:
        input_text += f"\n\nClarifications from user:\n{clarifications}"

        result = await Runner.run(planner_agent, input_text)
        print(f"Will perform {len(result.final_output.searches)} searches")
        return result.final_output


async def perform_searches(search_plan: WebSearchPlan):
    """ Perform the web searches according to the search plan, and return the results """
    print("Performing web searches...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Completed web searches")
    return results

async def search(item: WebSearchItem):
    """Use the search agent to run a web search for each item in the search plan"""
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

async def write_report(query: str, clarifications: str, search_results: list[str]):
    """Use the writer agent to write a report based on the search results"""
    print("THinking about report...")
    input = f"Original query: {query}\n\nClarifications: {clarifications}\n\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output


async def send_notification(report: ReportData):
    """Use the notication agent to send a notification with the report"""
    print("Sending notification....")
    result = await Runner.run(notification_agent, report.markdown_report)
    print("Notification sent")
    return report

async def send_email(report: ReportData):
    """Use the email agent to send an email with the report"""
    print("Sending email...")
    result = await Runner.run(email_agent,  report.markdown_report)
    print("Email sent")
    return report    

    
# Formatting functions
def format_questions(questions: ClarifyingQuestions) -> str:
    """ Format clarifying questions for dsiplay """
    lines = [
        "Awesome! I'd like to ask a few clarifying questions to help narrow the search to the relevant:\n"
    ]

    for i, q in enumerate(questions.questions, 1):
        lines.append(f"**{i}.  {q.question}?")

    lines.append("\nPlease answer these questions (You can answer all at once or just provide key points)")
    return "\n".join(lines)


# Check if user is answering the questions
def is_answering_questions(history) -> bool:
    if not history:
        return False
    last = history[-1]
    if isinstance(last, dict):
        last_content = last.get("content", "") or ""
    else:
        last_content = last[1] or ""  # tuple format
    return bool(last_content and "clarifying questions" in last_content.lower())


# Research flow

In [11]:
async def run(query: str, clarifications: str):
    """Run the research flow with clarification"""
    with trace("Deep Research trace"):
        print("Starting research...")
        search_plan = await plan_searches(query, clarifications)
        search_results = await perform_searches(search_plan)
        report = await write_report(query, clarifications, search_results)
        
        try:
            await send_notification(report)
        except Exception as e:
            print(f"❌ Notification failed: {e}")
        
        try:
            await send_email(report)
        except Exception as e:
            print(f"❌ Email failed: {e}")
        
        print("Hooray!")
        return report.markdown_report


In [ ]:
async def chat(message: str, history):
    if is_answering_questions(history):
        first = history[0]
        original_query = first["content"] if isinstance(first, dict) else first[0]

        yield "Great! Starting the research now... This may take a few seconds.\n\nPlanning searches..."
        try:
            report = await run(original_query, message)
            yield f"Research completed! 🎉\n\n{report}\n\nReport also sent to your email! 📧\n\nFeel free to ask me to research another topic!"
        except InputGuardrailTripwireTriggered as e:
            print(f"Exception attributes: {dir(e)}")
            yield "🚨 Sorry, this research topic is outside the allowed scope and cannot be processed.\n\nPlease try a different research topic!"
    else:
        questions = await get_clarifying_questions(message)
        yield format_questions(questions)

# Gradio UI

In [ ]:
ui = gr.ChatInterface(
    fn=chat,
    title="🔍 Deep Research Assistant",
    description="let me know what you'd like to research, I will demand clarification before diving in",
    examples=[
        "The impact of the war in Iran",
        "Latest AI Agent framework in 2026",
        "Best practice for RAG",
    ],
    theme=gr.themes.Soft(primary_hue="sky")
)

ui.launch(inbrowser=True)

/home/issoft/Workspace/agents/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Generating clarifying questions...
Generated 5 questions
Starting research...
Planning searches...
Exception attributes: ['__annotations__', '__cause__', '__class__', '__context__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__suppress_context__', '__traceback__', '__weakref__', 'add_note', 'args', 'guardrail_result', 'run_data', 'with_traceback']


Traceback (most recent call last):
  File "/tmp/ipykernel_7185/1459792328.py", line 8, in chat
    report = await run(original_query, message)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_7185/2108337016.py", line 5, in run
    search_plan = await plan_searches(query, clarifications)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_7185/2049123501.py", line 15, in plan_searches
    result = await Runner.run(planner_agent, input_text)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/issoft/Workspace/agents/.venv/lib/python3.12/site-packages/agents/run.py", line 296, in run
    return await runner.run(
           ^^^^^^^^^^^^^^^^^
  File "/home/issoft/Workspace/agents/.venv/lib/python3.12/site-packages/agents/run.py", line 548, in run
    input_guardrail_results, turn_result = await asyncio.gather(
                                           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/issoft/Workspace/agents/.venv/